In [ ]:

# AI801 Team Project Baseline
# Date: 6 Mar 2026
# Huy Nguyen 
# --------------------------------------

from __future__ import annotations

import argparse
import json
import math
import random
import sys
from collections import namedtuple
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Sequence

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    from IPython.display import Image as IPyImage, display
except Exception:
    IPyImage = None
    display = None

GameState = namedtuple('GameState', 'to_move utility board moves last_move')

PLAYER_X = 'X'
PLAYER_O = 'O'
EMPTY = '.'
ROWS = 6
COLS = 7
CONNECT_K = 4


def opponent(player: str) -> str:
    return PLAYER_O if player == PLAYER_X else PLAYER_X


class Game:
    initial = None

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, move):
        raise NotImplementedError

    def utility(self, state, player):
        raise NotImplementedError

    def terminal_test(self, state):
        return not self.actions(state)

    def is_terminal(self, state):
        return self.terminal_test(state)

    def to_move(self, state):
        return state.to_move

    def display(self, state):
        print(state)


class ConnectFour(Game):
    def __init__(self, rows: int = ROWS, cols: int = COLS, k: int = CONNECT_K):
        self.rows = rows
        self.cols = cols
        self.k = k
        board = tuple(tuple(EMPTY for _ in range(cols)) for _ in range(rows))
        self.initial = GameState(
            to_move=PLAYER_X,
            utility=0,
            board=board,
            moves=self._legal_moves(board),
            last_move=None,
        )

    def actions(self, state: GameState) -> Sequence[int]:
        return state.moves

    def result(self, state: GameState, move: int) -> GameState:
        if move not in state.moves:
            raise ValueError(f'Illegal move: {move}')
        row = self._drop_row(state.board, move)
        board_list = [list(r) for r in state.board]
        board_list[row][move] = state.to_move
        new_board = tuple(tuple(r) for r in board_list)
        utility = self._compute_utility(new_board, row, move, state.to_move)
        return GameState(
            to_move=opponent(state.to_move),
            utility=utility,
            board=new_board,
            moves=self._legal_moves(new_board) if utility == 0 else tuple(),
            last_move=move,
        )

    def utility(self, state: GameState, player: str) -> int:
        return state.utility if player == PLAYER_X else -state.utility

    def terminal_test(self, state: GameState) -> bool:
        return state.utility != 0 or len(state.moves) == 0

    def display(self, state: GameState) -> None:
        for row in state.board:
            print(' '.join(row))
        print(' '.join(str(c) for c in range(self.cols)))

    def winner(self, state: GameState) -> Optional[str]:
        if state.utility > 0:
            return PLAYER_X
        if state.utility < 0:
            return PLAYER_O
        return None

    def _legal_moves(self, board: Sequence[Sequence[str]]) -> tuple[int, ...]:
        legal = [c for c in range(self.cols) if board[0][c] == EMPTY]
        center = self.cols // 2
        legal.sort(key=lambda c: (abs(c - center), c))
        return tuple(legal)

    def _drop_row(self, board: Sequence[Sequence[str]], col: int) -> int:
        for r in range(self.rows - 1, -1, -1):
            if board[r][col] == EMPTY:
                return r
        raise ValueError(f'Column {col} is full.')

    def _compute_utility(self, board: Sequence[Sequence[str]], row: int, col: int, player: str) -> int:
        for dr, dc in ((1, 0), (0, 1), (1, 1), (1, -1)):
            total = 1
            total += self._count_direction(board, row, col, dr, dc, player)
            total += self._count_direction(board, row, col, -dr, -dc, player)
            if total >= self.k:
                return 1 if player == PLAYER_X else -1
        return 0

    def _count_direction(self, board: Sequence[Sequence[str]], row: int, col: int, dr: int, dc: int, player: str) -> int:
        count = 0
        r, c = row + dr, col + dc
        while 0 <= r < self.rows and 0 <= c < self.cols and board[r][c] == player:
            count += 1
            r += dr
            c += dc
        return count


PlayerFn = Callable[[ConnectFour, GameState], int]


def play_game(game: Game, strategies: Dict[str, Callable], verbose: bool = False):
    state = game.initial
    if verbose:
        game.display(state)
        print()
    while not game.terminal_test(state):
        current_player = game.to_move(state)
        if current_player not in strategies:
            raise KeyError(f'Missing strategy for player {current_player!r}.')
        move = strategies[current_player](game, state)
        if move not in game.actions(state):
            raise ValueError(f'Strategy for player {current_player!r} returned illegal move {move!r}.')
        state = game.result(state, move)
        if verbose:
            print(f'Player {current_player} move: {move}')
            game.display(state)
            print()
    return state


def random_player(game: ConnectFour, state: GameState) -> int:
    return random.choice(list(game.actions(state)))


def player(search_algorithm):
    return lambda game, state: search_algorithm(game, state)[1]


def query_player(game: ConnectFour, state: GameState) -> int:
    print('\nCurrent state:')
    game.display(state)
    print(f'Available moves: {list(game.actions(state))}')
    while True:
        raw = input(f'Player {state.to_move}, choose a column (0-{game.cols - 1}): ').strip()
        try:
            move = int(raw)
        except ValueError:
            print('Please enter an integer column number.')
            continue
        if move in game.actions(state):
            return move
        print('Illegal move. Choose one of:', list(game.actions(state)))


def cutoff_depth(limit: int):
    return lambda game, state, depth: depth > limit


def _score_window(window: Sequence[str], player: str) -> float:
    mine = window.count(player)
    theirs = window.count(opponent(player))
    empties = window.count(EMPTY)
    if mine > 0 and theirs > 0:
        return 0.0
    if mine == 4:
        return 100000.0
    if mine == 3 and empties == 1:
        return 120.0
    if mine == 2 and empties == 2:
        return 12.0
    if mine == 1 and empties == 3:
        return 1.0
    if theirs == 3 and empties == 1:
        return -150.0
    if theirs == 2 and empties == 2:
        return -10.0
    return 0.0


def connect4_heuristic(state: GameState, player: str) -> float:
    board = state.board
    rows = len(board)
    cols = len(board[0])
    if state.utility > 0:
        return 1_000_000.0 if player == PLAYER_X else -1_000_000.0
    if state.utility < 0:
        return 1_000_000.0 if player == PLAYER_O else -1_000_000.0
    score = 0.0
    center = cols // 2
    score += 4.0 * sum(1 for r in range(rows) if board[r][center] == player)
    score -= 4.0 * sum(1 for r in range(rows) if board[r][center] == opponent(player))
    for r in range(rows):
        for c in range(cols - 3):
            score += _score_window([board[r][c + i] for i in range(4)], player)
    for r in range(rows - 3):
        for c in range(cols):
            score += _score_window([board[r + i][c] for i in range(4)], player)
    for r in range(rows - 3):
        for c in range(cols - 3):
            score += _score_window([board[r + i][c + i] for i in range(4)], player)
    for r in range(rows - 3):
        for c in range(3, cols):
            score += _score_window([board[r + i][c - i] for i in range(4)], player)
    return score


def h_alphabeta_search(
    game: ConnectFour,
    state: GameState,
    cutoff: Optional[Callable[[ConnectFour, GameState, int], bool]] = None,
    h: Optional[Callable[[GameState, str], float]] = None,
):
    cutoff = cutoff or cutoff_depth(6)
    h = h or connect4_heuristic
    root_player = game.to_move(state)
    infinity = math.inf

    def max_value(s: GameState, alpha: float, beta: float, depth: int):
        if game.terminal_test(s):
            return float(game.utility(s, root_player)), None
        if cutoff(game, s, depth):
            return float(h(s, root_player)), None
        value = -infinity
        best_move = None
        for action in game.actions(s):
            child_value, _ = min_value(game.result(s, action), alpha, beta, depth + 1)
            if child_value > value:
                value = child_value
                best_move = action
                alpha = max(alpha, value)
            if value >= beta:
                break
        return value, best_move

    def min_value(s: GameState, alpha: float, beta: float, depth: int):
        if game.terminal_test(s):
            return float(game.utility(s, root_player)), None
        if cutoff(game, s, depth):
            return float(h(s, root_player)), None
        value = infinity
        best_move = None
        for action in game.actions(s):
            child_value, _ = max_value(game.result(s, action), alpha, beta, depth + 1)
            if child_value < value:
                value = child_value
                best_move = action
                beta = min(beta, value)
            if value <= alpha:
                break
        return value, best_move

    return max_value(state, -infinity, infinity, 0)


def get_ab_player(depth: int = 6) -> PlayerFn:
    return lambda game, state: h_alphabeta_search(game, state, cutoff=cutoff_depth(depth))[1]


def _terminal_score(game: ConnectFour, state: GameState, player: str) -> float:
    utility = game.utility(state, player)
    if utility > 0:
        return 1.0
    if utility < 0:
        return 0.0
    return 0.5


def _immediate_winning_move(game: ConnectFour, state: GameState, player: str) -> Optional[int]:
    for move in game.actions(state):
        if game.utility(game.result(state, move), player) > 0:
            return move
    return None


def _safe_moves(game: ConnectFour, state: GameState) -> List[int]:
    safe_moves: List[int] = []
    opp = opponent(state.to_move)
    for move in game.actions(state):
        child = game.result(state, move)
        opponent_can_win = False
        for reply in game.actions(child):
            if game.utility(game.result(child, reply), opp) > 0:
                opponent_can_win = True
                break
        if not opponent_can_win:
            safe_moves.append(move)
    return safe_moves


def _center_weighted_choice(game: ConnectFour, moves: Iterable[int], rng: random.Random) -> int:
    move_list = list(moves)
    if not move_list:
        raise ValueError('No moves to choose from.')
    center = game.cols // 2
    weighted: List[int] = []
    for move in move_list:
        weighted.extend([move] * max(1, game.cols - abs(move - center)))
    return rng.choice(weighted)


class MCTSNode:
    __slots__ = ('state', 'parent', 'move', 'player_just_moved', 'untried_moves', 'children', 'wins', 'visits')

    def __init__(self, game: ConnectFour, state: GameState, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.player_just_moved = opponent(state.to_move)
        self.untried_moves = list(game.actions(state))
        self.children: List["MCTSNode"] = []
        self.wins = 0.0
        self.visits = 0

    def select_child(self, exploration: float) -> "MCTSNode":
        log_parent = math.log(self.visits)

        def ucb(child: "MCTSNode") -> float:
            return (child.wins / child.visits) + exploration * math.sqrt(log_parent / child.visits)

        return max(self.children, key=ucb)

    def add_child(self, game: ConnectFour, move: int) -> "MCTSNode":
        child_state = game.result(self.state, move)
        child = MCTSNode(game, child_state, parent=self, move=move)
        self.untried_moves.remove(move)
        self.children.append(child)
        return child

    def update(self, result: float) -> None:
        self.visits += 1
        self.wins += result


class MCTSPlayer:
    def __init__(self, simulations: int = 1000, exploration: float = math.sqrt(2.0), seed: Optional[int] = None):
        if simulations < 1:
            raise ValueError('simulations must be >= 1')
        self.simulations = simulations
        self.exploration = exploration
        self.rng = random.Random(seed)

    def __call__(self, game: ConnectFour, state: GameState) -> int:
        immediate = _immediate_winning_move(game, state, state.to_move)
        if immediate is not None:
            return immediate
        root = MCTSNode(game, state)
        for _ in range(self.simulations):
            node = root
            rollout_state = state
            while not node.untried_moves and node.children:
                node = node.select_child(self.exploration)
                rollout_state = node.state
            if node.untried_moves and not game.terminal_test(rollout_state):
                move = _center_weighted_choice(game, node.untried_moves, self.rng)
                node = node.add_child(game, move)
                rollout_state = node.state
            terminal_state = self._rollout(game, rollout_state)
            while node is not None:
                node.update(_terminal_score(game, terminal_state, node.player_just_moved))
                node = node.parent
        if not root.children:
            return _center_weighted_choice(game, game.actions(state), self.rng)
        max_visits = max(child.visits for child in root.children)
        candidates = [child.move for child in root.children if child.visits == max_visits]
        return self.rng.choice(candidates)

    def _rollout(self, game: ConnectFour, state: GameState) -> GameState:
        current = state
        while not game.terminal_test(current):
            current = game.result(current, self._rollout_policy(game, current))
        return current

    def _rollout_policy(self, game: ConnectFour, state: GameState) -> int:
        win_now = _immediate_winning_move(game, state, state.to_move)
        if win_now is not None:
            return win_now
        safe_moves = _safe_moves(game, state)
        if safe_moves:
            return _center_weighted_choice(game, safe_moves, self.rng)
        return _center_weighted_choice(game, game.actions(state), self.rng)


def monte_carlo_tree_search(state: GameState, game: ConnectFour, N: int = 1000, seed: Optional[int] = None) -> int:
    return MCTSPlayer(simulations=N, seed=seed)(game, state)


def get_mcts_player(n_sims: int = 1000, seed: Optional[int] = None) -> PlayerFn:
    return lambda game, state: monte_carlo_tree_search(state, game, N=n_sims, seed=seed)


def _resolve_output_path(path_like: Path | str) -> Path:
    return Path(path_like).expanduser().resolve()


def print_header(title: str) -> None:
    line = '=' * len(title)
    print(f'\n{line}\n{title}\n{line}')


def print_kv(label: str, value) -> None:
    print(f'{label:<28} : {value}')


def print_table(title: str, rows: List[Dict[str, object]]) -> None:
    print_header(title)
    if not rows:
        print('No rows to display.')
        return
    headers = list(rows[0].keys())
    widths = {h: max(len(h), max(len(str(row[h])) for row in rows)) for h in headers}
    print(' | '.join(f'{h:{widths[h]}}' for h in headers))
    print('-+-'.join('-' * widths[h] for h in headers))
    for row in rows:
        print(' | '.join(f'{str(row[h]):{widths[h]}}' for h in headers))


def summarize_official(summary: Dict[str, object]) -> None:
    rounds = int(summary['rounds'])
    mcts_wins = int(summary['mcts_wins'])
    ab_wins = int(summary['alpha_beta_wins'])
    draws = int(summary['draws'])
    print_header('OFFICIAL TOURNAMENT SUMMARY')
    print_kv('Rounds', rounds)
    print_kv('MCTS simulations', summary['mcts_simulations'])
    print_kv('Alpha-Beta depth', summary['alpha_beta_depth'])
    print_kv('Seed', summary['seed'])
    print_kv('MCTS wins', f'{mcts_wins} ({mcts_wins / rounds:.1%})')
    print_kv('Alpha-Beta wins', f'{ab_wins} ({ab_wins / rounds:.1%})')
    print_kv('Draws', f'{draws} ({draws / rounds:.1%})')
    overall = 'MCTS' if mcts_wins > ab_wins else 'Alpha-Beta' if ab_wins > mcts_wins else 'Tie'
    print_kv('Overall winner', overall)


def summarize_balanced(summary: Dict[str, object]) -> None:
    print_header('BALANCED TOURNAMENT SUMMARY')
    print_kv('Rounds', summary['rounds'])
    print_kv('MCTS simulations', summary['mcts_simulations'])
    print_kv('Alpha-Beta depth', summary['alpha_beta_depth'])
    print_kv('Seed', summary['seed'])
    print_kv('MCTS points', summary['mcts_points'])
    print_kv('Alpha-Beta points', summary['alpha_beta_points'])
    rows: List[Dict[str, object]] = []
    for item in summary['results']:
        rows.append({
            'Round': item['round'],
            'MCTS side': 'X' if item['mcts_as_x'] else 'O',
            'Winner': item['winner'],
            'Winner side': item['winner_side'],
        })
    print_table('BALANCED ROUND RESULTS', rows)


def summarize_sweep(n_values: Sequence[int], results: Dict[int, List[float]]) -> None:
    print_header('PERFORMANCE SWEEP SUMMARY')
    depth_keys = sorted(int(d) for d in results.keys())
    headers = ['MCTS N'] + [f'AB depth {d}' for d in depth_keys]
    rows: List[List[str]] = []
    for idx, n in enumerate(n_values):
        row = [str(n)]
        for depth in depth_keys:
            row.append(f'{results[depth][idx]:.3f}')
        rows.append(row)
    widths = [max(len(headers[i]), max(len(row[i]) for row in rows)) for i in range(len(headers))]
    print(' | '.join(f'{headers[i]:{widths[i]}}' for i in range(len(headers))))
    print('-+-'.join('-' * widths[i] for i in range(len(headers))))
    for row in rows:
        print(' | '.join(f'{row[i]:{widths[i]}}' for i in range(len(headers))))
    print('\nInterpretation: score rate uses win = 1.0, draw = 0.5, loss = 0.0.')
    print('Values above 0.500 favor MCTS; below 0.500 favor Alpha-Beta.')


def maybe_display_plot(plot_path: Path, show_plot: bool) -> None:
    if show_plot and IPyImage is not None and display is not None and plot_path.exists():
        print_header('PLOT PREVIEW')
        display(IPyImage(filename=str(plot_path)))


def save_json(data: object, out_path: Path) -> Path:
    out_path = _resolve_output_path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(data, indent=2), encoding='utf-8')
    return out_path


def save_plot(n_values: Sequence[int], results: Dict[int, List[float]], out_path: Path) -> Path:
    out_path = _resolve_output_path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 6))
    for depth, row in sorted(results.items()):
        plt.plot(list(n_values), row, marker='o', label=f'Alpha-Beta Depth {depth}')
    plt.axhline(y=0.5, linestyle='--', label='Breakeven (0.5 score rate)')
    plt.title('MCTS Score Rate vs Alpha-Beta')
    plt.xlabel('MCTS Simulations (N)')
    plt.ylabel('MCTS Score Rate')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    return out_path


def run_human_vs_computer(ai: str = 'alphabeta', depth: int = 6, simulations: int = 1000, human_as: str = PLAYER_X):
    game = ConnectFour()
    if ai == 'alphabeta':
        computer = get_ab_player(depth)
    elif ai == 'mcts':
        computer = get_mcts_player(simulations)
    else:
        raise ValueError("ai must be 'alphabeta' or 'mcts'.")
    result = play_game(game, {human_as: query_player, opponent(human_as): computer}, verbose=True)
    print_header('GAME RESULT')
    print_kv('Human side', human_as)
    print_kv('Computer side', opponent(human_as))
    print_kv('Computer algorithm', ai)
    print_kv('Winner', game.winner(result) or 'Draw')
    return result


def run_official_tournament(rounds: int = 5, mcts_sims: int = 1000, ab_depth: int = 6, seed: int = 7, verbose: bool = True) -> Dict[str, object]:
    game = ConnectFour()
    rng = random.Random(seed)
    rows: List[Dict[str, object]] = []
    mcts_wins = 0
    ab_wins = 0
    draws = 0
    if verbose:
        print_header('RUNNING OFFICIAL TOURNAMENT')
        print_kv('Rounds', rounds)
        print_kv('MCTS side', 'X')
        print_kv('Alpha-Beta side', 'O')
        print_kv('MCTS simulations', mcts_sims)
        print_kv('Alpha-Beta depth', ab_depth)
        print_kv('Seed', seed)
    for round_no in range(1, rounds + 1):
        mcts_seed = rng.randrange(1_000_000_000)
        result = play_game(
            game,
            {PLAYER_X: get_mcts_player(mcts_sims, seed=mcts_seed), PLAYER_O: get_ab_player(ab_depth)},
            verbose=False,
        )
        if result.utility > 0:
            winner = 'MCTS'
            winner_side = 'X'
            mcts_wins += 1
        elif result.utility < 0:
            winner = 'Alpha-Beta'
            winner_side = 'O'
            ab_wins += 1
        else:
            winner = 'Draw'
            winner_side = '-'
            draws += 1
        rows.append({
            'Round': round_no,
            'Winner': winner,
            'Winner side': winner_side,
            'MCTS N': mcts_sims,
            'AB depth': ab_depth,
            'MCTS seed': mcts_seed,
        })
    summary: Dict[str, object] = {
        'rounds': rounds,
        'mcts_wins': mcts_wins,
        'alpha_beta_wins': ab_wins,
        'draws': draws,
        'mcts_simulations': mcts_sims,
        'alpha_beta_depth': ab_depth,
        'seed': seed,
        'round_details': rows,
    }
    if verbose:
        print_table('OFFICIAL TOURNAMENT ROUND RESULTS', rows)
        summarize_official(summary)
    return summary


def run_balanced_tournament(rounds: int = 6, mcts_sims: int = 1000, ab_depth: int = 6, seed: int = 11, verbose: bool = True) -> Dict[str, object]:
    game = ConnectFour()
    rng = random.Random(seed)
    mcts_points = 0.0
    ab_points = 0.0
    details: List[Dict[str, object]] = []
    if verbose:
        print_header('RUNNING BALANCED TOURNAMENT')
        print_kv('Rounds', rounds)
        print_kv('MCTS simulations', mcts_sims)
        print_kv('Alpha-Beta depth', ab_depth)
        print_kv('Seed', seed)
        print('MCTS alternates between X and O each round.')
    for index in range(rounds):
        round_no = index + 1
        mcts_as_x = index % 2 == 0
        mcts_seed = rng.randrange(1_000_000_000)
        mcts_player = get_mcts_player(mcts_sims, seed=mcts_seed)
        ab_player = get_ab_player(ab_depth)
        strategies = {
            PLAYER_X: mcts_player if mcts_as_x else ab_player,
            PLAYER_O: ab_player if mcts_as_x else mcts_player,
        }
        result = play_game(game, strategies, verbose=False)
        winner_side = game.winner(result) or '-'
        if winner_side == '-':
            winner = 'Draw'
            mcts_points += 0.5
            ab_points += 0.5
        else:
            mcts_won = (winner_side == PLAYER_X and mcts_as_x) or (winner_side == PLAYER_O and not mcts_as_x)
            if mcts_won:
                winner = 'MCTS'
                mcts_points += 1.0
            else:
                winner = 'Alpha-Beta'
                ab_points += 1.0
        details.append({
            'round': round_no,
            'mcts_as_x': mcts_as_x,
            'winner': winner,
            'winner_side': winner_side,
            'mcts_seed': mcts_seed,
        })
    summary: Dict[str, object] = {
        'rounds': rounds,
        'mcts_points': mcts_points,
        'alpha_beta_points': ab_points,
        'results': details,
        'mcts_simulations': mcts_sims,
        'alpha_beta_depth': ab_depth,
        'seed': seed,
    }
    if verbose:
        summarize_balanced(summary)
    return summary


def run_sweep(
    n_values: Sequence[int] = (10, 50, 200, 500),
    depth_values: Sequence[int] = (2, 4, 6),
    rounds_per_pair: int = 3,
    seed: int = 17,
    verbose: bool = True,
) -> Dict[int, List[float]]:
    game = ConnectFour()
    rng = random.Random(seed)
    results: Dict[int, List[float]] = {}
    if verbose:
        print_header('RUNNING PERFORMANCE SWEEP')
        print_kv('MCTS N values', list(n_values))
        print_kv('AB depth values', list(depth_values))
        print_kv('Rounds per pair', rounds_per_pair)
        print_kv('Seed', seed)
    for depth in depth_values:
        row: List[float] = []
        for n in n_values:
            score = 0.0
            for _ in range(rounds_per_pair):
                result = play_game(
                    game,
                    {PLAYER_X: get_mcts_player(n, seed=rng.randrange(1_000_000_000)), PLAYER_O: get_ab_player(depth)},
                    verbose=False,
                )
                if result.utility > 0:
                    score += 1.0
                elif result.utility == 0:
                    score += 0.5
            row.append(score / rounds_per_pair)
        results[int(depth)] = row
    if verbose:
        summarize_sweep(n_values, results)
    return results


def run_assignment_workflow(
    output_dir: Path = Path('./connect4_results'),
    official_rounds: int = 5,
    mcts_sims: int = 1000,
    ab_depth: int = 6,
    n_values: Sequence[int] = (10, 50, 200, 500),
    depth_values: Sequence[int] = (2, 4, 6),
    sweep_rounds: int = 3,
    seed: int = 23,
    show_plot: bool = False,
    verbose: bool = True,
) -> Dict[str, object]:
    output_dir = _resolve_output_path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    official = run_official_tournament(official_rounds, mcts_sims, ab_depth, seed, verbose=verbose)
    balanced = run_balanced_tournament(max(official_rounds, 6), mcts_sims, ab_depth, seed + 1, verbose=verbose)
    sweep = run_sweep(n_values, depth_values, sweep_rounds, seed + 2, verbose=verbose)
    official_json = save_json(official, output_dir / 'official_tournament_summary.json')
    balanced_json = save_json(balanced, output_dir / 'balanced_tournament_summary.json')
    sweep_json = save_json({'n_values': list(n_values), 'results': sweep}, output_dir / 'performance_sweep.json')
    plot_path = save_plot(n_values, sweep, output_dir / 'mcts_ab_performance.png')
    print_header('FILES SAVED')
    print_kv('Output directory', output_dir)
    print_kv('Official summary JSON', official_json)
    print_kv('Balanced summary JSON', balanced_json)
    print_kv('Sweep summary JSON', sweep_json)
    print_kv('Plot PNG', plot_path)
    maybe_display_plot(plot_path, show_plot)
    return {
        'official': official,
        'balanced': balanced,
        'sweep': sweep,
        'output_dir': str(output_dir),
        'official_json': str(official_json),
        'balanced_json': str(balanced_json),
        'sweep_json': str(sweep_json),
        'plot_path': str(plot_path),
    }


def print_assignment_sections() -> None:
    print_header('ASSIGNMENT TASKS')
    print('1. Human (X) vs Alpha-Beta Cutoff (O)')
    print('2. Official 5-round tournament: MCTS (N=1000) vs Alpha-Beta (depth 6)')
    print('3. Performance sweep over MCTS N and Alpha-Beta depth')
    print('4. Save readable summaries and the comparison plot')


def _normalize_argv(argv: Optional[Sequence[str]] = None) -> List[str]:
    raw = list(sys.argv[1:] if argv is None else argv)
    valid_commands = {'demo-human', 'tournament', 'balanced', 'sweep', 'assignment', 'all'}
    if not raw:
        return ['all']
    if raw[0] in valid_commands:
        return raw
    if any('ipykernel' in part or 'jupyter' in part or part.endswith('.json') for part in raw):
        return ['all']
    return raw


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description='Single-file Connect Four project with readable summaries and explicit save paths.')
    sub = parser.add_subparsers(dest='command', required=True)

    demo = sub.add_parser('demo-human', help='Human vs Alpha-Beta or MCTS.')
    demo.add_argument('--ai', choices=['alphabeta', 'mcts'], default='alphabeta')
    demo.add_argument('--depth', type=int, default=6)
    demo.add_argument('--simulations', type=int, default=1000)
    demo.add_argument('--human-as', choices=[PLAYER_X, PLAYER_O], default=PLAYER_X)

    tournament = sub.add_parser('tournament', help='Run the official fixed-side tournament.')
    tournament.add_argument('--rounds', type=int, default=5)
    tournament.add_argument('--mcts-sims', type=int, default=1000)
    tournament.add_argument('--ab-depth', type=int, default=6)
    tournament.add_argument('--seed', type=int, default=7)
    tournament.add_argument('--output-json', type=Path, default=Path('./connect4_results/official_tournament_summary.json'))

    balanced = sub.add_parser('balanced', help='Run the optional balanced tournament.')
    balanced.add_argument('--rounds', type=int, default=6)
    balanced.add_argument('--mcts-sims', type=int, default=1000)
    balanced.add_argument('--ab-depth', type=int, default=6)
    balanced.add_argument('--seed', type=int, default=11)
    balanced.add_argument('--output-json', type=Path, default=Path('./connect4_results/balanced_tournament_summary.json'))

    sweep = sub.add_parser('sweep', help='Run the performance sweep and save the plot.')
    sweep.add_argument('--n-values', type=int, nargs='+', default=[10, 50, 200, 500])
    sweep.add_argument('--depth-values', type=int, nargs='+', default=[2, 4, 6])
    sweep.add_argument('--rounds', type=int, default=3)
    sweep.add_argument('--seed', type=int, default=17)
    sweep.add_argument('--output-json', type=Path, default=Path('./connect4_results/performance_sweep.json'))
    sweep.add_argument('--plot-path', type=Path, default=Path('./connect4_results/mcts_ab_performance.png'))
    sweep.add_argument('--show-plot', action='store_true')

    assignment = sub.add_parser('assignment', help='Run the full assignment workflow with readable output.')
    assignment.add_argument('--output-dir', type=Path, default=Path('./connect4_results'))
    assignment.add_argument('--show-plot', action='store_true')

    all_cmd = sub.add_parser('all', help='Run all automated deliverables.')
    all_cmd.add_argument('--rounds', type=int, default=5)
    all_cmd.add_argument('--mcts-sims', type=int, default=1000)
    all_cmd.add_argument('--ab-depth', type=int, default=6)
    all_cmd.add_argument('--n-values', type=int, nargs='+', default=[10, 50, 200, 500])
    all_cmd.add_argument('--depth-values', type=int, nargs='+', default=[2, 4, 6])
    all_cmd.add_argument('--sweep-rounds', type=int, default=3)
    all_cmd.add_argument('--seed', type=int, default=23)
    all_cmd.add_argument('--output-dir', type=Path, default=Path('./connect4_results'))
    all_cmd.add_argument('--show-plot', action='store_true')

    return parser


def main(argv: Optional[Sequence[str]] = None) -> None:
    args = build_parser().parse_args(_normalize_argv(argv))

    if args.command == 'demo-human':
        run_human_vs_computer(ai=args.ai, depth=args.depth, simulations=args.simulations, human_as=args.human_as)
        return

    if args.command == 'tournament':
        summary = run_official_tournament(args.rounds, args.mcts_sims, args.ab_depth, args.seed, verbose=True)
        output_path = save_json(summary, args.output_json)
        print_header('FILE SAVED')
        print_kv('Official summary JSON', output_path)
        return

    if args.command == 'balanced':
        summary = run_balanced_tournament(args.rounds, args.mcts_sims, args.ab_depth, args.seed, verbose=True)
        output_path = save_json(summary, args.output_json)
        print_header('FILE SAVED')
        print_kv('Balanced summary JSON', output_path)
        return

    if args.command == 'sweep':
        results = run_sweep(args.n_values, args.depth_values, args.rounds, args.seed, verbose=True)
        json_path = save_json({'n_values': args.n_values, 'results': results}, args.output_json)
        plot_path = save_plot(args.n_values, results, args.plot_path)
        print_header('FILES SAVED')
        print_kv('Sweep summary JSON', json_path)
        print_kv('Plot PNG', plot_path)
        maybe_display_plot(plot_path, args.show_plot)
        return

    if args.command == 'assignment':
        print_assignment_sections()
        run_assignment_workflow(
            output_dir=args.output_dir,
            show_plot=args.show_plot,
            verbose=True,
        )
        return

    print_assignment_sections()
    run_assignment_workflow(
        output_dir=args.output_dir,
        official_rounds=args.rounds,
        mcts_sims=args.mcts_sims,
        ab_depth=args.ab_depth,
        n_values=args.n_values,
        depth_values=args.depth_values,
        sweep_rounds=args.sweep_rounds,
        seed=args.seed,
        show_plot=args.show_plot,
        verbose=True,
    )


if __name__ == '__main__':
    main()


ASSIGNMENT TASKS
1. Human (X) vs Alpha-Beta Cutoff (O)
2. Official 5-round tournament: MCTS (N=1000) vs Alpha-Beta (depth 6)
3. Performance sweep over MCTS N and Alpha-Beta depth
4. Save readable summaries and the comparison plot

RUNNING OFFICIAL TOURNAMENT
Rounds                       : 5
MCTS side                    : X
Alpha-Beta side              : O
MCTS simulations             : 1000
Alpha-Beta depth             : 6
Seed                         : 23
